In [0]:
# Create widgets
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format", "", "Source File Format")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
# Build the landing path
landing_path = f"/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# Check whether the path (and its parent directories) exists and contains files
try:
    landing_files = dbutils.fs.ls(landing_path)
except Exception as e:
    if "FileNotFoundException" in str(e) or "does not exist" in str(e).lower() or "not found" in str(e).lower():
        landing_files = None
    else:
        raise

if not landing_files:
    msg = f"No new files are received at {landing_path}"
    print(msg)
    dbutils.notebook.exit(msg)  # exits the notebook successfully (no failure)

In [0]:
if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql.functions import lit, current_timestamp
updated_df = landing_df.withColumn("landing_timestamp", lit(landing_timestamp)) \
                      .withColumn("insert_timestamp", current_timestamp())

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")